<a href="https://colab.research.google.com/github/rikharhm0208-coder/PROJECT_RIKHA/blob/main/MetodeLanczos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 **Simulator Algoritma Lanczos**
"Halo! Selamat datang di aplikasi simulasi Algoritma Lanczos. Ingin tahu seberapa cepat nilai eigen matriks Anda konvergen?

**Cara Penggunaan:**
1. Silakan **running** program di bawah ini dengan menekan tombol play (▶️).
2. Kemudian masukkan **ukuran** dan **elemen matriks** Anda di kolom sebelah kiri.
3. Sesuaikan **jumlah iterasi**, dan lihat hasilnya pada tabel serta grafik yang tersedia.

**Selamat mencoba!"**
---

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd

# --- 1. Fungsi Algoritma Lanczos ---
def lanczos(A, k, u1=None):
    n = A.shape[0]
    # Menggunakan vektor awal konstan (ones) agar hasil konsisten saat parameter diubah
    u = np.ones(n) / np.sqrt(n) if u1 is None else u1.copy()
    u = u / np.linalg.norm(u)
    U = np.zeros((n, k))
    alpha, beta = np.zeros(k), np.zeros(k-1)
    U[:, 0] = u
    for j in range(k):
        v = A @ U[:, j]
        if j > 0: v -= beta[j-1] * U[:, j-1]
        alpha[j] = np.dot(U[:, j], v)
        v -= alpha[j] * U[:, j]
        # Full Reorthogonalization (Mencegah kehilangan ortogonalitas)
        for i in range(j): v -= np.dot(U[:, i], v) * U[:, i]
        if j < k - 1:
            beta[j] = np.linalg.norm(v)
            if beta[j] < 1e-12: break
            U[:, j+1] = v / beta[j]
    T = np.diag(alpha)
    for j in range(k-1): T[j, j+1] = T[j+1, j] = beta[j]
    return T

# --- 2. Komponen Antarmuka (Widgets) ---
style = {'description_width': 'initial'}

n_input = widgets.IntText(value=3, description='Ukuran Matriks (n):', style=style)
matrix_area = widgets.Textarea(
    value='4 1 0\n1 3 1\n0 1 2',
    placeholder='Masukkan elemen matriks (spasi untuk kolom, enter untuk baris)',
    description='Elemen Matriks:',
    style=style, layout={'height': '120px', 'width': '300px'}
)
k_slider = widgets.IntSlider(value=3, min=1, max=10, description='Dimensi Krylov (k):', style=style)
btn_plot = widgets.Button(description="Hitung & Plot", button_style='success', icon='check', layout={'width': '300px'})
output_area = widgets.Output()

# Widget Petunjuk Penggunaan (HTML)
petunjuk_html = widgets.HTML(
    value="""
    <div style="background-color: #ffffff; padding: 20px; border-radius: 10px; border: 1px solid #ddd; box-shadow: 2px 2px 5px rgba(0,0,0,0.05); margin-left: 20px; min-width: 350px;">
        <h4 style="margin-top: 0; color: #2e59a8; border-bottom: 2px solid #2e59a8; padding-bottom: 5px;">
            📘 PETUNJUK PENGGUNAAN
        </h4>
        <ul style="font-size: 13px; color: #444; line-height: 1.6; padding-left: 20px;">
            <li><b>Langkah 1:</b> Masukkan ukuran matriks persegi (n).</li>
            <li><b>Langkah 2:</b> Ketik elemen matriks. Pisahkan antar angka dengan <b>spasi</b> dan antar baris dengan <b>Enter</b>.</li>
            <li><b>Langkah 3:</b> Geser slider <b>k</b> untuk menentukan jumlah iterasi.</li>
            <li><b>Langkah 4:</b> Klik tombol hijau untuk melihat hasil.</li>
        </ul>
        <div style="background-color: #fff4e5; padding: 10px; border-radius: 5px; font-size: 12px; border-left: 4px solid #ffa000;">
            ⚠️ <b>Syarat:</b> Matriks harus <b>Simetris</b> (A = Aᵀ).
        </div>
    </div>
    """,
    layout=widgets.Layout(flex='1')
)

# --- 3. Fungsi Logika Utama ---
def proses_simulasi(b):
    with output_area:
        clear_output(wait=True)
        try:
            # Parsing input matriks
            raw_data = matrix_area.value.strip().split('\n')
            A = np.array([list(map(float, row.split())) for row in raw_data])

            # Validasi Input
            if A.shape[0] != n_input.value or A.shape[1] != n_input.value:
                st_err = f"⚠️ Error: Ukuran matriks ({A.shape[0]}x{A.shape[1]}) tidak sesuai dengan n={n_input.value}"
                display(widgets.HTML(f"<p style='color:red;'>{st_err}</p>"))
                return

            if not np.allclose(A, A.T):
                display(widgets.HTML("<p style='color:orange;'>⚠️ Peringatan: Matriks tidak sepenuhnya simetris!</p>"))

            # Perhitungan
            true_eigs = np.linalg.eigvalsh(A)
            k_val = min(k_slider.value, n_input.value)

            res_list = []
            approx_min, approx_max = [], []

            for k in range(1, k_val + 1):
                T = lanczos(A, k)
                eig_T = np.linalg.eigvalsh(T)
                approx_min.append(eig_T[0])
                approx_max.append(eig_T[-1])
                res_list.append([k, f"{eig_T[0]:.6f}", f"{eig_T[-1]:.6f}"])

            # Tampilan Tabel
            df = pd.DataFrame(res_list, columns=['Iterasi (k)', 'λ_min', 'λ_max'])
            display(widgets.HTML("<h3>📊 Hasil Numerik Iterasi:</h3>"))
            display(df.style.hide(axis='index'))

            # Tampilan Grafik
            display(widgets.HTML("<br><h3>📈 Grafik Konvergensi:</h3>"))
            fig, ax = plt.subplots(figsize=(10, 5))
            ax.plot(range(1, k_val+1), approx_min, 'bo-', label=r'Hampiran $\lambda_{min}$')
            ax.plot(range(1, k_val+1), approx_max, 'rs-', label=r'Hampiran $\lambda_{max}$')
            ax.axhline(true_eigs[0], color='blue', ls='--', alpha=0.5, label=r'$\lambda_{min}$ Sebenarnya')
            ax.axhline(true_eigs[-1], color='red', ls='--', alpha=0.5, label=r'$\lambda_{max}$ Sebenarnya')

            ax.set_xlabel("Iterasi (k)")
            ax.set_ylabel("Nilai Eigen")
            ax.grid(True, alpha=0.3)
            ax.legend()
            plt.show()

        except Exception as e:
            display(widgets.HTML(f"<p style='color:red;'>⚠️ Kesalahan Input: {str(e)}</p>"))

btn_plot.on_click(proses_simulasi)

# --- 4. Tata Letak Akhir (Layout) ---
kolom_kiri = widgets.VBox([n_input, matrix_area, k_slider, btn_plot])
baris_atas = widgets.HBox([kolom_kiri, petunjuk_html], layout=widgets.Layout(align_items='stretch'))
st_header = widgets.HTML("<h1 style='color: #2e59a8;'>📊 Dashboard Lanczos Interaktif</h1><hr>")

# Menampilkan semua ke layar
display(widgets.VBox([st_header, baris_atas, output_area]))

# Jalankan simulasi awal
proses_simulasi(None)